In [1]:
from lcb_runner.benchmarks import code_generation

dataset = code_generation.load_code_generation_dataset(release_version="release_v5", start_date="2024-01-01", diffulty=code_generation.Difficulty.HARD)

/Users/kaushitha/Documents/APR/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using 'test' split of the dataset.
Filtered problems by difficulty: hard
Loaded 194 problems


In [2]:
dataset[0]

CodeGenerationProblem(question_title='maximize-the-number-of-partitions-after-operations', question_content='You are given a 0-indexed string s and an integer k.\nYou are to perform the following partitioning operations until s is empty:\n\nChoose the longest prefix of s containing at most k distinct characters.\nDelete the prefix from s and increase the number of partitions by one. The remaining characters (if any) in s maintain their initial order.\n\nBefore the operations, you are allowed to change at most one index in s to another lowercase English letter.\nReturn an integer denoting the maximum number of resulting partitions after the operations by optimally choosing at most one index to change.\n \nExample 1:\n\nInput: s = "accca", k = 2\nOutput: 3\nExplanation: In this example, to maximize the number of resulting partitions, s[2] can be changed to \'b\'.\ns becomes "acbca".\nThe operations can now be performed as follows until s becomes empty:\n- Choose the longest prefix contai

In [3]:
from common.llm_client import create_llm_client, LLMClient
import common.code_preprocessing as cpp

sample_problem: str = dataset[0].question_content
starter_code: str = dataset[0].starter_code 

if len(starter_code) == 0:
    starter_code = """
class Solution:
    def sol(self, input_str):
"""


llm_client: LLMClient  = create_llm_client(provider='openai', model='gpt-5')
print(f"LLM Reasoning Effort: {llm_client.reasoning_effort}")

print("Sample Problem: ")
print(sample_problem)

print("Starter Code: ")
print(starter_code)

print("=== Generating Code ===")

INITIAL_CODE_POPULATION_SIZE = 10
initial_code_prompt = f"Write {INITIAL_CODE_POPULATION_SIZE} distinct solutions to solve the following problem: {sample_problem}\n```python\n{starter_code}```. Each solution should be in a separate code block."
response = llm_client.generate(initial_code_prompt)


# formatting response
code_blocks = cpp.parsers.extract_all_code_blocks_from_response(response)

if len(code_blocks) != INITIAL_CODE_POPULATION_SIZE:
    raise ValueError(f"Expected {INITIAL_CODE_POPULATION_SIZE} code blocks, but got {len(code_blocks)}\nResponse: {response}")


for code_idx, code in enumerate(code_blocks):
    print(f"Code Block {code_idx+1}:\n{code}\n")

LLM Reasoning Effort: minimal
Sample Problem: 
You are given a 0-indexed string s and an integer k.
You are to perform the following partitioning operations until s is empty:

Choose the longest prefix of s containing at most k distinct characters.
Delete the prefix from s and increase the number of partitions by one. The remaining characters (if any) in s maintain their initial order.

Before the operations, you are allowed to change at most one index in s to another lowercase English letter.
Return an integer denoting the maximum number of resulting partitions after the operations by optimally choosing at most one index to change.
 
Example 1:

Input: s = "accca", k = 2
Output: 3
Explanation: In this example, to maximize the number of resulting partitions, s[2] can be changed to 'b'.
s becomes "acbca".
The operations can now be performed as follows until s becomes empty:
- Choose the longest prefix containing at most 2 distinct characters, "acbca".
- Delete the prefix, and s becomes 

In [6]:
INITIAL_TEST_POPULATION_SIZE = 20
print("=== Generating Tests ===")
initial_test_prompt = f"Write {INITIAL_TEST_POPULATION_SIZE} distinct unit tests in a Python code block for the following problem:\n{sample_problem}\nThe solution is imported in the format:\n```python\n{starter_code}```\nWrite the tests using unittest framework. Do not use the examples givn in the problem."
print(initial_test_prompt)

response = llm_client.generate(initial_test_prompt)
print(f"response:\n{response}\n")
# formatting response
test_block = cpp.parsers.extract_code_block_from_response(response)

if not test_block:
    raise ValueError(f"Expected a test class block, but none found.\nResponse: {response}")

test_cases = cpp.analyzers.extract_test_methods_code(test_block)
if len(test_cases) != INITIAL_TEST_POPULATION_SIZE:
    raise ValueError(f"Expected {INITIAL_TEST_POPULATION_SIZE} test cases, but got {len(test_cases)}\nTest Block: {test_block}")

print(f"Test Block:\n{test_block}\n")

=== Generating Tests ===
Write 20 distinct unit tests in a Python code block for the following problem:
You are given a 0-indexed string s and an integer k.
You are to perform the following partitioning operations until s is empty:

Choose the longest prefix of s containing at most k distinct characters.
Delete the prefix from s and increase the number of partitions by one. The remaining characters (if any) in s maintain their initial order.

Before the operations, you are allowed to change at most one index in s to another lowercase English letter.
Return an integer denoting the maximum number of resulting partitions after the operations by optimally choosing at most one index to change.
 
Example 1:

Input: s = "accca", k = 2
Output: 3
Explanation: In this example, to maximize the number of resulting partitions, s[2] can be changed to 'b'.
s becomes "acbca".
The operations can now be performed as follows until s becomes empty:
- Choose the longest prefix containing at most 2 distinct

In [ ]:
from common.sandbox import create_safe_test_environment
import numpy as np

INITIAL_CODE_POPULATION_SIZE = len(code_blocks)
INITIAL_TEST_POPULATION_SIZE = len(test_cases)
matrix = np.zeros((INITIAL_CODE_POPULATION_SIZE, INITIAL_TEST_POPULATION_SIZE), dtype=int)
sandbox = create_safe_test_environment()

for code_idx, code in enumerate(code_blocks):
    print(f"Executing Code Block {code_idx+1}: ")
    script = cpp.builders.build_test_script_for_lcb(code, test_block)
    test_results = sandbox.execute_test_script(script)
    for test_idx, test in enumerate(test_results.test_results):
        print(f"Test: {test.name}, Status: {test.status}, Details: {test.details}")
        if test.status == "passed":
            matrix[code_idx, test_idx] = 1
        elif test.status == "failed":
            matrix[code_idx, test_idx] = 0  # failed
        else:
            matrix[code_idx, test_idx] = 0  # Let's treat errors as failures for now

Executing Code Block 1: 
Test: test_all_same_k1, Status: failed, Details: self.assertEqual(self.sol.maxPartitionsAfterOperations('aaaaaa', 1), 6)
AssertionError: 3 != 6
Test Description: Test method: test_all_same_k1
Test: test_all_same_large_k, Status: passed, Details: None
Test Description: Test method: test_all_same_large_k
Test: test_alternating_two_k1, Status: passed, Details: None
Test Description: Test method: test_alternating_two_k1
Test: test_alternating_two_k2, Status: failed, Details: self.assertEqual(self.sol.maxPartitionsAfterOperations('ababab', 2), 1)
AssertionError: 3 != 1
Test Description: Test method: test_alternating_two_k2
Test: test_cluster_k2, Status: failed, Details: self.assertEqual(self.sol.maxPartitionsAfterOperations('aabccdee', 2), 4)
AssertionError: 3 != 4
Test Description: Test method: test_cluster_k2
Test: test_complex_mix_k2, Status: failed, Details: self.assertEqual(self.sol.maxPartitionsAfterOperations('cabacab', 2), 3)
AssertionError: 4 != 3
Test Desc

In [ ]:
matrix

array([[1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0],
       [1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0],
       [1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0],
       [1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0],
       [1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0],
       [1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0],
       [1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0],
       [1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0],
       [1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0],
       [1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0]])

In [ ]:
from common.coevolution import CoevolutionConfig, initialize_populations, update_population_beliefs

config = CoevolutionConfig(
    initial_code_population_size=INITIAL_CODE_POPULATION_SIZE,
    initial_test_population_size=INITIAL_TEST_POPULATION_SIZE,
    c0_prior=0.6,
    t0_prior=0.6, 
    alpha=0.01,
    beta=0.2,
    gamma=0.2,
    learning_rate=0.1
)

# Updated to match the new function signature (returns only 2 arrays)
code_probs, test_probs = initialize_populations(config)
print("Initial Code Probabilities:")
print([round(a, 4) for a in code_probs.tolist()])
print("Initial Test Probabilities:")
print([round(a, 4) for a in test_probs.tolist()])

new_code_probs, new_test_probs = update_population_beliefs(code_probs, test_probs, matrix, config)


print("=== Updated Probabilities ===")
print("Code:")
print([round(a, 4) for a in new_code_probs.tolist()])

print("Test:")
print([round(a, 4) for a in new_test_probs.tolist()])


print("\n\n=== Comparison with Pass Rates ===")
print("Pass rates for each code snippet:")
# in matrix rows are code snippets and columns are tests, so we take mean across axis 1
pass_rates = matrix.mean(axis=1)
print([round(a, 4) for a in pass_rates.tolist()])

print("Pass rates for each test:")
print([round(a, 4) for a in matrix.mean(axis=0).tolist()])

Initial Code Probabilities:
[0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6]
Initial Test Probabilities:
[0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6]
=== Updated Probabilities ===
Code:
[0.8221, 0.8221, 0.8221, 0.8221, 0.8221, 0.8221, 0.8221, 0.8221, 0.8221, 0.8221]
Test:
[0.9222, 0.9222, 0.3443, 0.9222, 0.9222, 0.3443, 0.9222, 0.9222, 0.9222, 0.9222, 0.9222, 0.9222, 0.3443, 0.3443, 0.3443, 0.9222, 0.9222, 0.9222, 0.9222, 0.3443]


=== Comparison with Pass Rates ===
Pass rates for each code snippet:
[0.7, 0.7, 0.7, 0.7, 0.7, 0.7, 0.7, 0.7, 0.7, 0.7]
Pass rates for each test:
[1.0, 1.0, 0.0, 1.0, 1.0, 0.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 1.0, 0.0]
